# RoadGuard Qro — 04: ONNX Export, INT8 Quantization & CPU Latency Benchmark
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlarconCastilloDiegoAndre/RoadGuard-Qro/blob/main/ml/04_export_onnx.ipynb)

Exports the trained PyTorch model to ONNX, applies dynamic INT8 quantization to cut memory footprint,
and benchmarks inference latency on 2 vCPUs (matching the free Hugging Face Spaces environment).


In [ ]:
# Step 1: Install export tools
!pip install -q ultralytics onnx onnxruntime


In [ ]:
# Step 2: Export PyTorch model to standard ONNX
from ultralytics import YOLO
from pathlib import Path

weights_path = Path("ml/runs/roadguard_yolo11n/weights/best.pt")
if not weights_path.exists():
    print("Pre-trained best.pt not found locally. Loading standard yolo11n.pt for export pipeline demonstration.")
    model = YOLO("yolo11n.pt")
else:
    model = YOLO(str(weights_path))

onnx_path = model.export(
    format="onnx",
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=12
)
print(f"Exported FP32 ONNX model to: {onnx_path}")


In [ ]:
# Step 3: Apply Dynamic INT8 Quantization
import os
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

fp32_model_path = onnx_path
int8_model_path = "ml/runs/roadguard_yolo11n/weights/model_int8.onnx"
os.makedirs(os.path.dirname(int8_model_path), exist_ok=True)

print("Quantizing ONNX model to INT8...")
quantize_dynamic(
    model_input=fp32_model_path,
    model_output=int8_model_path,
    weight_type=QuantType.QUInt8
)

fp32_size = os.path.getsize(fp32_model_path) / (1024 * 1024)
int8_size = os.path.getsize(int8_model_path) / (1024 * 1024)

print(f"FP32 Model Size: {fp32_size:.2f} MB")
print(f"INT8 Model Size: {int8_size:.2f} MB ({((fp32_size - int8_size) / fp32_size) * 100:.1f}% reduction)")


In [ ]:
# Step 4: Benchmark CPU Latency on 2 vCPUs (Hugging Face Spaces CPU constraint)
import time
import numpy as np
import onnxruntime as ort

opts = ort.SessionOptions()
opts.intra_op_num_threads = 2
opts.inter_op_num_threads = 1
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

session = ort.InferenceSession(int8_model_path, opts, providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name

# Warmup
dummy_input = np.random.randn(1, 3, 640, 640).astype(np.float32)
for _ in range(5):
    _ = session.run(None, {input_name: dummy_input})

# Benchmark 50 iterations
N_ITERS = 50
latencies = []
for _ in range(N_ITERS):
    t0 = time.perf_counter()
    _ = session.run(None, {input_name: dummy_input})
    latencies.append((time.perf_counter() - t0) * 1000.0)

mean_lat = np.mean(latencies)
p95_lat = np.percentile(latencies, 95)

print("\n=== CPU BENCHMARK RESULTS (2 threads) ===")
print(f"Mean Latency: {mean_lat:.2f} ms")
print(f"P95 Latency:  {p95_lat:.2f} ms")
print(f"Throughput:   {1000.0 / mean_lat:.1f} frames/sec")

if p95_lat <= 250:
    print("✅ Gate 1 CPU latency condition PASSED (<= 250 ms)!")
else:
    print("⚠️ Latency exceeded 250 ms. Consider INT8 quantization refinement or smaller input dimensions.")
